<a href="https://colab.research.google.com/github/yy726/GPQA-eval/blob/submission-yy726/notebooks/11_sft_grpo_unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/submission-yy726/notebooks/11_sft_grpo_unsloth.ipynb)

# 11 · SFT + GRPO on Qwen3（照 Unsloth 官方 notebook 的写法）

结构参考 Unsloth 的两个官方 Colab：
[Qwen3 (4B) Instruct（SFT）](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-Instruct.ipynb) 和
[Qwen3 (4B) GRPO](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-GRPO.ipynb)：
同一个模型对象开 `fast_inference=True`（内置 vLLM），先做一小段 **SFT 预热**（学会格式），再做 **GRPO**（可验证奖励）。

我们在这个流程上加了**同一个 vLLM 引擎里的 GPQA 快速评测**，在三个时间点各测一次：
`0_base`（不加 LoRA）→ `1_sft`（SFT LoRA）→ `2_grpo`（GRPO LoRA），直接看到每一步的变化。

| 模式 | 数据量 | 评测 | 预计用时（T4，0.6B，待实测） |
|---|---|---|---|
| `quick` | SFT 200 条 Dolci + 200 条选择题格式样本，40 步；GRPO 30 步 | GPQA Diamond 前 60 题，最长 1024 token | 约 15–25 分钟 |
| `full` | SFT 3000 + 1000 条，1 epoch；GRPO 300 步 | GPQA Diamond 全部 198 题，最长 2048 token | 约 1.5–3 小时 |

⚠️ 这里的快速评测**不是**排行榜分数：题目子集更小、生成更短、选项顺序与 lm-eval 不同。它只用来比较同一次运行里 base / SFT / GRPO 的相对变化。
正式分数请把合并后的模型推到 Hub，再在新的 runtime 里用 `main` 分支的 `01_eval_gpqa.ipynb` 评测。

**模型大小怎么选**（Qwen 官方后训练版的 GPQA Diamond，随机猜是 25%）：

| 模型 | 官方 non-thinking | 官方 thinking | 用途 |
|---|---|---|---|
| Qwen3-0.6B | 22.9 | 27.9 | 只适合**验证流程**：官方完整后训练都只在随机水平附近，能看到的主要是格式变化 |
| Qwen3-1.7B | 28.6 | 40.1 | **能看到变化的最小尺寸**，提升主要来自长推理 |
| Qwen3-4B | 41.7 | 55.9 | 变化最明显；Unsloth 的官方 T4 GRPO 示例就是这个尺寸 |

In [1]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

name, memory.total [MiB], compute_cap
Tesla T4, 15360 MiB, 7.5


In [2]:
# 克隆提交分支（数据准备 / 去污染脚本在里面）
import os
if not os.path.exists("GPQA-eval"):
    !git clone -q -b submission-yy726 https://github.com/yy726/GPQA-eval.git
%cd GPQA-eval
!git pull -q

/content/GPQA-eval


In [3]:
# %%capture
# # 安装：照搬 Unsloth 官方 GRPO notebook 的 Colab 安装方式（T4 用 vllm==0.11.2）。约 3–5 分钟。
# import os
# os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # Unsloth：GRPO 时让 vLLM 和训练共享显存
# !pip install --upgrade -qqq uv
# try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
# except: _numpy = "numpy"; _pil = "pillow"
# try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
# except: is_t4 = False
# _vllm, _triton = ('vllm==0.11.2', 'triton') if is_t4 else ('vllm==0.15.1', 'triton')
# !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
# !uv pip install -qqq {_triton}
# try:
#     import importlib.metadata as _md; _torch_v = tuple(int(_p) for _p in _md.version("torch").split("+")[0].split(".")[:2])
# except Exception:
#     _torch_v = ()
# _torchao = "torchao>=0.16.0" if _torch_v >= (2, 10) else "torchao>=0.16.0,<0.18.0"
# !uv pip install -qqq --no-deps --upgrade "{_torchao}"
# !uv pip install -qqq transformers==4.57.6
# !uv pip install -qqq --no-deps trl==0.22.2

In [4]:
import os
# 禁用 FlashInfer 相关的采样器和后端
os.environ["VLLM_USE_FLASHINFER"] = "0"
os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

print("VLLM_USE_FLASHINFER_SAMPLER:", os.environ.get("VLLM_USE_FLASHINFER_SAMPLER"))
print("VLLM_ATTENTION_BACKEND:", os.environ.get("VLLM_ATTENTION_BACKEND"))



VLLM_USE_FLASHINFER_SAMPLER: 0
VLLM_ATTENTION_BACKEND: None


In [5]:
import os, importlib.metadata as md
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"   # %%capture 那一格里设的环境变量在这里再设一次，确保生效
VERSIONS = {}
for p in ["unsloth", "vllm", "trl", "transformers", "torch", "peft"]:
    try: VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError: VERSIONS[p] = None
print(VERSIONS)

{'unsloth': '2026.11.3', 'vllm': '0.33.0', 'trl': '1.15.0', 'transformers': '5.17.0', 'torch': '2.14.0', 'peft': '0.21.2'}


In [6]:
# GPQA 是 gated 数据集：先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款；Colab 左侧 🔑 里放 HF_TOKEN
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

## 0. 配置

In [7]:
MODE         = "quick"                        # "quick"（验证流程）| "full"
MODEL_NAME   = "unsloth/Qwen3-0.6B-Base"      # 看到真实变化：换成 "unsloth/Qwen3-1.7B-Base" 或 "unsloth/Qwen3-4B-Base"
BASE_MODEL   = "Qwen/Qwen3-0.6B-Base"         # 提交时填的原始 base（与 MODEL_NAME 对应）
LOAD_IN_4BIT = False                          # 16-bit LoRA（Unsloth 官方 GRPO 示例也是 False）。
                                              # 想用 bnb-4bit：MODEL_NAME="unsloth/Qwen3-0.6B-Base-unsloth-bnb-4bit"，这里改 True
MAX_SEQ_LEN  = 2048
LORA_RANK    = 32
GPU_UTIL     = 0.7                            # vLLM 占用比例；OOM 就调小（4B 可参考 Unsloth 的 0.9 + standby）

DOLCI_DATASET = "allenai/Dolci-Instruct-SFT-7B"   # 也可以试 "allenai/Dolci-Think-SFT-7B"（思维链更长，会被长度过滤掉很多）
DOLCI_INCLUDE = []                                # 例如 ["math", "science"]；空 = 不按来源过滤（最快）
DOLCI_EXCLUDE = ["safety", "wildguard"]

Q = dict(
    quick=dict(sft_dolci_n=200, sft_mcq_n=200, sft_steps=40, grpo_prompts=200, grpo_steps=30,
               eval_n=60, eval_max_tokens=1024, shuffle_buffer=1000),
    full=dict(sft_dolci_n=3000, sft_mcq_n=1000, sft_steps=-1, grpo_prompts=3000, grpo_steps=300,
              eval_n=None, eval_max_tokens=2048, shuffle_buffer=20000),
)[MODE]
print(MODE, Q)

quick {'sft_dolci_n': 200, 'sft_mcq_n': 200, 'sft_steps': 40, 'grpo_prompts': 200, 'grpo_steps': 30, 'eval_n': 60, 'eval_max_tokens': 1024, 'shuffle_buffer': 1000}


## 1. 加载模型（`fast_inference=True` = 训练和生成共用一个 vLLM 引擎）

In [8]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = LOAD_IN_4BIT,
    fast_inference = True,          # 内置 vLLM，用于 GRPO 采样和下面的快速评测
    max_lora_rank = LORA_RANK,
    gpu_memory_utilization = GPU_UTIL,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_RANK * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
# 用 Qwen3 自带的 ChatML 模板（锁定评测 lm-eval 也会用保存下来的这个模板），不像 Unsloth GRPO 示例那样换成自定义标签
if tokenizer.chat_template is None:
    from unsloth.chat_templates import get_chat_template
    tokenizer = get_chat_template(tokenizer, chat_template = "qwen3-instruct")
print(tokenizer.apply_chat_template([{"role": "user", "content": "Hi"}], tokenize = False, add_generation_prompt = True))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1110 04:09:30.317000 11316 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 04:09:30.420000 11316 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 04:09:35.759000 11316 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1110 04:09:35.764000 11316 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1110 04:09:35.769000 11316 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.3: Fast Qwen3 patching. Transformers: 5.17.0. vLLM: 0.33.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not planning a device map; vLLM places its own weights. Using `sequential`.
Unsloth: Standby mode is enabled. Changing `gpu_memory_utilization` to 0.76.
Unsloth: Patched vLLM's piecewise graph-partition pass so compilation_config=3 (CUDA graphs) works with LoRA.
Unsloth: Not an error, but `level` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `use_cudagraph` is not supported in vLLM.config.CompilationConfig. Skippin

/usr/local/lib/python3.13/dist-packages/vllm/engine/arg_utils.py:1992: UserWarning: --disable-cascade-attn is deprecated and has no effect since cascade attention has been removed. It will be removed in v0.33.0.
  warnings.warn(_DISABLE_CASCADE_ATTN_DEPRECATION_MESSAGE, stacklevel=1)


(MainProcess pid=11316) INFO 11-10 04:10:36 [core.py:139] Initializing a V1 LLM engine (v0.33.0) with config: model='/root/.cache/huggingface/hub/models--unsloth--Qwen3-0.6B-Base/snapshots/bb9061da4d541093b20453a403e205510037d5a9', speculative_config=None, tokenizer='/root/.cache/huggingface/hub/models--unsloth--Qwen3-0.6B-Base/snapshots/bb9061da4d541093b20453a403e205510037d5a9', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


... (10 outputs left out)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['layer_norm1', 'cross_attn_input_layernorm', 'post_per_layer_input_norm', 'pre_feedforward_layernorm', 'k_norm', 'post_feedforward_layernorm', 'norm', 'post_feedforward_layernorm_1', 'norm2', 'q_norm', 'operator_norm', 'ffn_norm', 'pre_feedforward_layernorm_2', 'input_layernorm', 'post_layernorm', 'post_feedforward_layernorm_2', 'k_layernorm', 'q_layernorm', 'post_attention_layernorm', 'cross_attn_post_attention_layernorm', 'attention_norm', 'layer_norm2', 'norm1']


Unsloth 2026.11.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen3DecoderLayer).
<|im_start|>user
Hi<|im_end|>
<|im_start|>assistant



## 2. GPQA 快速评测（同一个 vLLM 引擎，只输出聚合数字）

题面与 lm-eval 的 `gpqa_diamond_cot_zeroshot` 模板一致；答案提取用两种方式：
`strict` = 出现 `The answer is (X)`，`flex` = 输出中最后一个 `(A)`–`(D)`（排行榜用的就是 flex）。

In [10]:
import random, re, json, statistics
from collections import Counter
from datasets import load_dataset
from vllm import SamplingParams

TEMPLATE = ("What is the correct answer to this question:{q}\nChoices:\n"
            "(A) {a}\n(B) {b}\n(C) {c}\n(D) {d}\nLet's think step by step: ")
STRICT_RE = re.compile(r"The answer is \(([A-D])\)")
LETTER_RE = re.compile(r"\(([A-D])\)")

def _clean(t):
    return " ".join(str(t).replace(" [title]", ". ").split())

def build_gpqa(config, n=None):
    ds = load_dataset("Idavidrein/gpqa", config, split = "train")
    rows = []
    for i, d in enumerate(ds):
        choices = [_clean(d["Incorrect Answer 1"]), _clean(d["Incorrect Answer 2"]),
                   _clean(d["Incorrect Answer 3"]), _clean(d["Correct Answer"])]
        correct = choices[3]
        random.Random(1000 + i).shuffle(choices)
        rows.append((TEMPLATE.format(q = _clean(d["Question"]), a = choices[0], b = choices[1],
                                     c = choices[2], d = choices[3]), "ABCD"[choices.index(correct)]))
    return rows[:n] if n else rows

GPQA = build_gpqa("gpqa_diamond", Q["eval_n"])
GPQA_TEXTS = [tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize = False,
                                            add_generation_prompt = True) for p, _ in GPQA]
RESULTS = {}

def quick_eval(tag, lora_request = None):
    sp = SamplingParams(temperature = 0.6, top_p = 0.95, max_tokens = Q["eval_max_tokens"], seed = 0)
    outs = model.fast_generate(GPQA_TEXTS, sampling_params = sp, lora_request = lora_request)
    strict = flex = extracted = truncated = 0
    lengths, letters = [], Counter()
    for o, (_, gold) in zip(outs, GPQA):
        c = o.outputs[0]
        m = STRICT_RE.findall(c.text)
        f = LETTER_RE.findall(c.text)
        strict += bool(m) and m[-1] == gold
        flex += bool(f) and f[-1] == gold
        extracted += bool(f)
        truncated += c.finish_reason == "length"
        lengths.append(len(c.token_ids))
        letters[f[-1] if f else "none"] += 1
    n = len(GPQA)
    RESULTS[tag] = dict(
        n = n, acc_flex = round(100 * flex / n, 1), acc_strict = round(100 * strict / n, 1),
        extract_rate = round(100 * extracted / n, 1), truncation_rate = round(100 * truncated / n, 1),
        mean_tokens = round(statistics.fmean(lengths)), letters = dict(sorted(letters.items())),
    )
    print(tag, RESULTS[tag])

quick_eval("0_base")

gpqa_diamond.csv:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/198 [00:00<?, ? examples/s]

(MainProcess pid=11316) INFO 11-10 04:16:08 [scheduler.py:3074] setting pause state to UNPAUSED


Rendering prompts:   0%|          | 0/60 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/60 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(MainProcess pid=11316) WARNING 11-10 04:16:08 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _topp_sb_stats_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(MainProcess pid=11316) WARNING 11-10 04:16:10 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _topp_sb_step_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(MainProcess pid=11316) WARNING 11-10 04:16:12 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _topp_sb_mask_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 60/60 [01:47<00:00,  1.79s/it, est. speed input: 120.86 toks/s, output: 391.30 toks/s]

0_base {'n': 60, 'acc_flex': 16.7, 'acc_strict': 0.0, 'extract_rate': 60.0, 'truncation_rate': 56.7, 'mean_tokens': 699, 'letters': {'A': 7, 'B': 6, 'C': 11, 'D': 12, 'none': 24}}


## 3. SFT 预热（照 Unsloth Instruct notebook：chat 模板转成 `text` + `train_on_responses_only`）

两部分数据：
1. **Dolci**（OLMo 3 的 SFT 数据）：流式读取、按长度过滤到 `MAX_SEQ_LEN/2` 以内（和 Unsloth GRPO 示例一样）。
2. **选择题格式样本**：SciQ 训练集，题面套用 GPQA 模板，答案是 SciQ 的 `support` 段落 + `The answer is (X).`。
   这部分教会模型评测要的作答格式。SciQ 与 GPQA 无关，但仍然走一遍去污染检查。

In [13]:
import re

# Let's inspect or directly patch scripts/prepare_dolci_sft.py to set a fallback chat template if not present
script_path = "scripts/prepare_dolci_sft.py"
with open(script_path, "r") as f:
    content = f.read()

# We'll inject code to check tokenizer.chat_template and set fallback to 'qwen3-instruct' template if it is empty
setup_patch = """
    from unsloth.chat_templates import get_chat_template
    if getattr(tok, 'chat_template', None) is None:
        tok = get_chat_template(tok, chat_template='qwen3-instruct')
"""

# Find where AutoTokenizer is loaded or where `tok` is defined in prepare_dolci_sft.py
# We will add fallback template setup right after tokenizer is initialized
if 'tok =' in content and 'get_chat_template' not in content:
    patched_content = re.sub(
        r"(tok\s*=\s*AutoTokenizer\.from_pretrained\(.*\))",
        r"\1\n" + setup_patch,
        content
    )
    with open(script_path, "w") as f:
        f.write(patched_content)
    print("Successfully patched scripts/prepare_dolci_sft.py with fallback chat template!")
else:
    print("Script might already be patched or contains different structure.")

Successfully patched scripts/prepare_dolci_sft.py with fallback chat template!


In [14]:
import os
os.makedirs("data", exist_ok = True)
inc = " ".join(DOLCI_INCLUDE); exc = " ".join(DOLCI_EXCLUDE)
inc_arg = f"--include {inc}" if inc else ""
N_DOLCI, HALF_LEN, SHUF = Q["sft_dolci_n"], MAX_SEQ_LEN // 2, Q["shuffle_buffer"]
!python scripts/prepare_dolci_sft.py --dataset {DOLCI_DATASET} --out data/dolci.jsonl --n {N_DOLCI} --tokenizer {MODEL_NAME} --max-tokens {HALF_LEN} --shuffle-buffer {SHUF} {inc_arg} --exclude {exc}

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:2161: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1110 04:20:44.286000 14588 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supporte

In [16]:
import os
# Ensure the data directory exists both globally and in the current working directory
print("Current working directory:", os.getcwd())
os.makedirs("data", exist_ok=True)

# Let's verify that the output of scripts/prepare_dolci_sft.py was generated correctly
expected_path = "data/dolci.jsonl"
if os.path.exists(expected_path):
    print(f"Success! '{expected_path}' was found. Size: {os.path.getsize(expected_path)} bytes.")
else:
    print(f"Error: '{expected_path}' was not found in the current working directory!")
    # If it was generated in /content/data or elsewhere, let's locate it
    for root, dirs, files in os.walk('.'):
        if 'dolci.jsonl' in files:
            found_path = os.path.join(root, 'dolci.jsonl')
            print(f"Found dolci.jsonl at: {found_path}")

Current working directory: /content/GPQA-eval
Success! 'data/dolci.jsonl' was found. Size: 318815 bytes.


In [17]:
SCIQ = load_dataset("allenai/sciq", split = "train").shuffle(seed = 0)

def sciq_mcq(ex, idx):
    rng = random.Random(idx)
    choices = [ex["distractor1"], ex["distractor2"], ex["distractor3"], ex["correct_answer"]]
    rng.shuffle(choices)
    letter = "ABCD"[choices.index(ex["correct_answer"])]
    prompt = TEMPLATE.format(q = ex["question"], a = choices[0], b = choices[1], c = choices[2], d = choices[3])
    return prompt, letter

# SciQ 前一段给 SFT，后一段给 GRPO，互不重叠
sft_mcq = []
for i, ex in enumerate(SCIQ):
    if len(sft_mcq) >= Q["sft_mcq_n"]:
        break
    if ex["support"].strip():
        prompt, letter = sciq_mcq(ex, i)
        sft_mcq.append({"messages": [{"role": "user", "content": prompt},
                                     {"role": "assistant", "content": ex["support"].strip() + f"\n\nThe answer is ({letter})."}]})
SCIQ_GRPO_START = i

dolci = [json.loads(l) for l in open("data/dolci.jsonl")]
sft_rows = [{"messages": r["prompt"] + r["completion"]} for r in dolci] + sft_mcq
with open("data/sft_all.jsonl", "w") as f:
    for r in sft_rows:
        f.write(json.dumps(r, ensure_ascii = False) + "\n")
!python scripts/decontam.py --train-jsonl data/sft_all.jsonl --out data/decontam_sft.json > /dev/null
bad = set(json.load(open("data/decontam_sft.json"))["flagged_row_indices"])
sft_rows = [r for i, r in enumerate(sft_rows) if i not in bad]
random.Random(0).shuffle(sft_rows)
DATA = dict(dolci = len(dolci), sciq_format = len(sft_mcq), dropped_decontam = len(bad))
print(DATA)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
gpqa_extended.csv: 100% 4.09M/4.09M [00:00<00:00, 12.5MB/s]
Generating train split: 100% 546/546 [00:00<00:00, 2445.84 examples/s]
{'dolci': 200, 'sciq_format': 200, 'dropped_decontam': 0}


In [18]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

sft_ds = Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize = False)} for r in sft_rows])
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_steps = 5,
        max_steps = Q["sft_steps"],
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        output_dir = "outputs/sft",
    ),
)
# 只在 assistant 回复上算 loss（Qwen3 ChatML 标记）
trainer = train_on_responses_only(trainer, instruction_part = "<|im_start|>user\n",
                                  response_part = "<|im_start|>assistant\n")
sft_stats = trainer.train()
sft_losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
SFT = dict(steps = trainer.state.global_step, minutes = round(sft_stats.metrics["train_runtime"] / 60, 1),
           loss_first = sft_losses[0] if sft_losses else None, loss_last = sft_losses[-1] if sft_losses else None)
print(SFT)

model.save_lora("sft_lora")
quick_eval("1_sft", model.load_lora("sft_lora"))

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/400 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 400 | Num Epochs = 1 | Total steps = 40
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 20,185,088 of 616,235,008 (3.28% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
(MainProcess pid=11316) WARNING 11-10 05:11:37 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _rms_layernorm_forward. This causes a latency spike; consider extending warmup to cover this shape/config.
(MainProcess pid=11316) WARNING 11-10 05:11:37 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _qk_norm_rope_forward. This causes a latency spike; consider extending warmup to cover this shape/config.
(MainProcess pid=11316) WARNING 11-10 05:11:37 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _fg_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(MainProcess pid=11316) WARNING 11-10 05:11:39 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _fl_partial_kernel. This causes a latency spike; con

Step,Training Loss
5,1.348117
10,1.504139
15,1.455938
20,1.398228
25,1.283256
30,1.512594
35,1.432772
40,1.370265


... (5 outputs left out)

Processed prompts:   0%|          | 0/60 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(MainProcess pid=11316) WARNING 11-10 05:12:37 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _lora_shrink_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 60/60 [03:51<00:00,  3.85s/it, est. speed input: 56.05 toks/s, output: 265.73 toks/s]

1_sft {'n': 60, 'acc_flex': 18.3, 'acc_strict': 18.3, 'extract_rate': 88.3, 'truncation_rate': 100.0, 'mean_tokens': 1024, 'letters': {'A': 17, 'B': 9, 'C': 10, 'D': 17, 'none': 7}}


## 4. GRPO（照 Unsloth GRPO notebook：`vllm_sampling_params` + 多个奖励函数）

题目：SciQ 中没用于 SFT 的部分，题面同样套 GPQA 模板。奖励：
- `format_reward`：出现且只出现一次 `The answer is (X)` → +1.0
- `correctness_reward`：最后一个 `(A)`–`(D)` 等于标准答案 → +2.0；输出里完全没有选项字母 → −1.0

In [19]:
import numpy as np
del sft_ds, trainer
torch.cuda.empty_cache(); import gc; gc.collect()

grpo_rows = []
for j in range(SCIQ_GRPO_START, len(SCIQ)):
    if len(grpo_rows) >= Q["grpo_prompts"]:
        break
    prompt, letter = sciq_mcq(SCIQ[j], j)
    grpo_rows.append({"prompt": [{"role": "user", "content": prompt}], "answer": letter})
grpo_ds = Dataset.from_list(grpo_rows)

lens = [len(tokenizer.apply_chat_template(r["prompt"], add_generation_prompt = True, tokenize = True)) for r in grpo_rows]
max_prompt_length = int(np.quantile(lens, 0.9)) + 1
grpo_ds = grpo_ds.select(np.where(np.array(lens) < max_prompt_length)[0])
max_completion_length = min(1024, MAX_SEQ_LEN - max_prompt_length)
print(len(grpo_ds), "prompts; max_prompt_length", max_prompt_length, "max_completion_length", max_completion_length)

def format_reward(completions, **kwargs):
    return [1.0 if len(STRICT_RE.findall(c[0]["content"])) == 1 else 0.0 for c in completions]

def correctness_reward(completions, answer, **kwargs):
    out = []
    for c, a in zip(completions, answer):
        found = LETTER_RE.findall(c[0]["content"])
        out.append(-1.0 if not found else (2.0 if found[-1] == a else 0.0))
    return out

180 prompts; max_prompt_length 71 max_completion_length 1024


In [20]:
from trl import GRPOConfig, GRPOTrainer

vllm_sampling_params = SamplingParams(
    min_p = 0.1, top_p = 1.0, top_k = -1, seed = 3407,
    stop = [tokenizer.eos_token, "<|im_end|>"],
    include_stop_str_in_output = True,
)
NUM_GEN = 8
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = NUM_GEN,     # 每步 = 2 道题 × 8 个采样
    gradient_accumulation_steps = 2,
    num_generations = NUM_GEN,
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = Q["grpo_steps"],
    save_steps = 10_000,
    report_to = "none",
    output_dir = "outputs/grpo",
)
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [format_reward, correctness_reward],
    args = training_args,
    train_dataset = grpo_ds,
)
grpo_stats = trainer.train()

def _avg(key, part):
    vals = [h[key] for h in trainer.state.log_history if key in h]
    k = max(1, len(vals) // 5)
    return round(float(np.mean(vals[:k] if part == "first" else vals[-k:])), 3) if vals else None

GRPO = dict(steps = trainer.state.global_step, minutes = round(grpo_stats.metrics["train_runtime"] / 60, 1))
for key in ["reward", "rewards/format_reward/mean", "rewards/correctness_reward/mean", "completion_length",
            "completions/mean_length"]:
    if any(key in h for h in trainer.state.log_history):
        GRPO[key] = {"first": _avg(key, "first"), "last": _avg(key, "last")}
print(GRPO)

model.save_lora("grpo_lora")
quick_eval("2_grpo", model.load_lora("grpo_lora"))

Unsloth: GRPO now defaults to TRL's loss_type = 'dapo' with beta = 0.0. Set loss_type = 'bnpo', beta = 0.001 for Unsloth's previous default.
Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.
Unsloth: `max_prompt_length` is not supported by the installed TRL's GRPOConfig and will be IGNORED - filter overlong prompts out of your dataset before training instead.
(MainProcess pid=11316) INFO 11-10 05:16:26 [scheduler.py:3074] setting pause state to PAUSED_NEW
(MainProcess pid=11316) INFO 11-10 05:16:26 [sleep_mode_backend.py:202] Using sleep-mode backend: cumem


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 180 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 20,185,088 of 616,235,008 (3.28% trained)


(MainProcess pid=11316) INFO 11-10 05:16:27 [scheduler.py:3074] setting pause state to UNPAUSED
(MainProcess pid=11316) INFO 11-10 05:16:27 [scheduler.py:3074] setting pause state to UNPAUSED
(MainProcess pid=11316) WARNING 11-10 05:16:33 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _exact_merge_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
Unsloth: Merged-weight rollouts on (all, automatic): folding the adapter into 196 shared weights; exact fold / unfold, no pristine copy (only non-round-tripping elements are kept, in pinned host memory).
Unsloth: exact merged rollout keeps 3,805,076 of 440,401,920 weights (0.86%) for the restore.
(MainProcess pid=11316) INFO 11-10 05:16:49 [scheduler.py:3074] setting pause state to UNPAUSED
(MainProcess pid=11316) WARNING 11-10 05:16:50 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _topk_log_softmax_kernel. This causes a latency spike; consider extending warm

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward / mean,rewards / format_reward / std,rewards / correctness_reward / mean,rewards / correctness_reward / std
1,0.000000,1.250000,1.000000,1024.000000,1024.000000,1024.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.250000,1.000000
2,0.000000,1.437500,1.030776,1024.000000,1024.000000,1024.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.062500,0.250000,1.375000,0.957427
3,-0.000000,0.500000,0.894427,1024.000000,1024.000000,1024.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.894427
4,-0.000000,1.312500,1.078193,1024.000000,1024.000000,1024.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.062500,0.250000,1.250000,1.000000
5,0.000000,0.687500,0.946485,1024.000000,1024.000000,1024.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.062500,0.250000,0.625000,0.957427
6,-0.000000,0.937500,1.123610,1024.000000,1024.000000,1024.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.062500,0.250000,0.875000,1.024695
7,0.000000,0.750000,1.000000,1024.000000,1024.000000,1024.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.750000,1.000000
8,0.000000,1.000000,1.032796,1024.000000,1024.000000,1024.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.032796
9,-0.000000,0.625000,0.957427,1024.000000,1024.000000,1024.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.625000,0.957427
10,0.000000,1.000000,1.032796,1024.000000,1024.000000,1024.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.032796


(MainProcess pid=11316) INFO 11-10 05:17:59 [scheduler.py:3074] setting pause state to UNPAUSED
(MainProcess pid=11316) INFO 11-10 05:17:59 [scheduler.py:3074] setting pause state to UNPAUSED
(MainProcess pid=11316) INFO 11-10 05:17:59 [scheduler.py:3074] setting pause state to UNPAUSED
(MainProcess pid=11316) INFO 11-10 05:18:36 [scheduler.py:3074] setting pause state to PAUSED_NEW
(MainProcess pid=11316) INFO 11-10 05:18:43 [scheduler.py:3074] setting pause state to UNPAUSED
(MainProcess pid=11316) INFO 11-10 05:18:43 [scheduler.py:3074] setting pause state to UNPAUSED
Unsloth: exact merged rollout keeps 3,803,739 of 440,401,920 weights (0.86%) for the restore.
(MainProcess pid=11316) INFO 11-10 05:18:43 [scheduler.py:3074] setting pause state to UNPAUSED
(MainProcess pid=11316) INFO 11-10 05:19:19 [scheduler.py:3074] setting pause state to PAUSED_NEW
(MainProcess pid=11316) INFO 11-10 05:19:26 [scheduler.py:3074] setting pause state to UNPAUSED
(MainProcess pid=11316) INFO 11-10 05:

Unsloth: Restored added_tokens_decoder metadata in outputs/grpo/checkpoint-30/tokenizer_config.json.


{'steps': 30, 'minutes': 22.2, 'reward': {'first': 1.021, 'last': 1.062}, 'rewards/format_reward/mean': {'first': 0.042, 'last': 0.021}, 'rewards/correctness_reward/mean': {'first': 0.979, 'last': 1.042}, 'completion_length': {'first': 1024.0, 'last': 1024.0}, 'completions/mean_length': {'first': 1024.0, 'last': 1024.0}}
(MainProcess pid=11316) INFO 11-10 05:38:44 [scheduler.py:3074] setting pause state to UNPAUSED


Rendering prompts:   0%|          | 0/60 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 60/60 [03:50<00:00,  3.84s/it, est. speed input: 56.31 toks/s, output: 266.93 toks/s]

2_grpo {'n': 60, 'acc_flex': 18.3, 'acc_strict': 18.3, 'extract_rate': 81.7, 'truncation_rate': 100.0, 'mean_tokens': 1024, 'letters': {'A': 10, 'B': 7, 'C': 7, 'D': 25, 'none': 11}}


## 5. 汇总（把最后打印的 JSON 整段贴回给 Claude）

In [21]:
print(f"{'stage':8s} {'acc_flex':>8s} {'acc_strict':>10s} {'extract%':>9s} {'trunc%':>7s} {'tokens':>7s}")
for k, v in RESULTS.items():
    print(f"{k:8s} {v['acc_flex']:8.1f} {v['acc_strict']:10.1f} {v['extract_rate']:9.1f} {v['truncation_rate']:7.1f} {v['mean_tokens']:7d}")
REPORT = dict(notebook = "11_sft_grpo_unsloth", mode = MODE, model = MODEL_NAME, load_in_4bit = LOAD_IN_4BIT,
              q = Q, dolci = DOLCI_DATASET, data = DATA, sft = SFT, grpo = GRPO, quick_eval = RESULTS, versions = VERSIONS)
os.makedirs("outputs", exist_ok = True)
json.dump(REPORT, open("outputs/report_11.json", "w"), indent = 1)
print("=" * 30, "把下面整段 JSON 贴回给 Claude", "=" * 30)
print(json.dumps(REPORT, ensure_ascii = False))

stage    acc_flex acc_strict  extract%  trunc%  tokens
0_base       16.7        0.0      60.0    56.7     699
1_sft        18.3       18.3      88.3   100.0    1024
2_grpo       18.3       18.3      81.7   100.0    1024
============================== 把下面整段 JSON 贴回给 Claude ==============================
{"notebook": "11_sft_grpo_unsloth", "mode": "quick", "model": "unsloth/Qwen3-0.6B-Base", "load_in_4bit": false, "q": {"sft_dolci_n": 200, "sft_mcq_n": 200, "sft_steps": 40, "grpo_prompts": 200, "grpo_steps": 30, "eval_n": 60, "eval_max_tokens": 1024, "shuffle_buffer": 1000}, "dolci": "allenai/Dolci-Instruct-SFT-7B", "data": {"dolci": 200, "sciq_format": 200, "dropped_decontam": 0}, "sft": {"steps": 40, "minutes": 1.0, "loss_first": 1.3481167793273925, "loss_last": 1.3702654838562012}, "grpo": {"steps": 30, "minutes": 22.2, "reward": {"first": 1.021, "last": 1.062}, "rewards/format_reward/mean": {"first": 0.042, "last": 0.021}, "rewards/correctness_reward/mean": {"first": 0.979, "last": 1

## 6.（可选）合并保存 → 推到 Hub → 用 `main` 分支的 `01` 做正式评测

`merged_16bit` 会把 LoRA 合并进权重并存成 fp16（排行榜要求）。推到 Hub 之后，
**换一个新的 runtime** 打开 [`01_eval_gpqa.ipynb`](https://colab.research.google.com/github/yy726/GPQA-eval/blob/main/notebooks/01_eval_gpqa.ipynb)，
`MODEL` 填你的仓库名、`BASE` 填上面的 `BASE_MODEL`。

In [22]:
PUSH = False
HUB_REPO = "YOUR_HF_USERNAME/" + MODEL_NAME.split("/")[-1].lower() + "-dolci-sft-grpo"
if PUSH:
    model.push_to_hub_merged(HUB_REPO, tokenizer, save_method = "merged_16bit", token = os.environ.get("HF_TOKEN"))
    print(f"pushed: https://huggingface.co/{HUB_REPO}")
else:
    model.save_pretrained_merged("outputs/merged_16bit", tokenizer, save_method = "merged_16bit")
    print("saved locally: outputs/merged_16bit")

Unsloth: Restored added_tokens_decoder metadata in outputs/merged_16bit/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `outputs/merged_16bit`: 100%|██████████| 1/1 [00:33<00:00, 33.78s/it]


Successfully copied all 1 files from cache to `outputs/merged_16bit`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:52<00:00, 52.31s/it]


Unsloth: Merge process complete. Saved to `/content/GPQA-eval/outputs/merged_16bit`
saved locally: outputs/merged_16bit
